<a href="https://colab.research.google.com/github/dzzd97/LIO-SAM/blob/master/studio/Unsloth_Studio_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth Studio on your local device, follow [our guide](https://unsloth.ai/docs/new/unsloth-studio/install). Unsloth Studio is licensed [AGPL-3.0](https://github.com/unslothai/unsloth/blob/main/studio/LICENSE.AGPL-3.0).

### Unsloth Studio

Train and run open models with [**Unsloth Studio**](https://unsloth.ai/docs/new/unsloth-studio/start). NEW! Installation should now only take 2 mins!


We are actively working on making Unsloth Studio install on Colab T4 GPUs faster.

[Features](https://unsloth.ai/docs/new/unsloth-studio#features) • [Quickstart](https://unsloth.ai/docs/new/unsloth-studio/start) • [Data Recipes](https://unsloth.ai/docs/new/unsloth-studio/data-recipe) • [Studio Chat](https://unsloth.ai/docs/new/unsloth-studio/chat) • [Export](https://unsloth.ai/docs/new/unsloth-studio/export)

<p align="left"><img src="https://github.com/unslothai/unsloth/raw/main/studio/frontend/public/studio%20github%20landscape%20colab%20display.png" width="600"></p>

### Setup: Clone repo and run setup

In [ ]:
!git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
!chmod +x studio/setup.sh && ./studio/setup.sh

### Start Unsloth Studio

In [ ]:
import sys, time
sys.path.insert(0, "/content/unsloth/studio/backend")
from colab import start
start()

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8888, height = 1200, width = "100%")
for _ in range(10000): time.sleep(300), print("=", end = "")

In [1]:
# 安装 Unsloth 及依赖
# Uninstall existing torch components to ensure a clean installation of compatible versions
!pip uninstall -y torch torchvision torchaudio

# Install a compatible torch version first to avoid conflicts
# unsloth requires torch<2.11.0,>=2.4.0. We'll use torch 2.10.0 based on previous dependency resolution attempts.
# Compatible torchvision and torchaudio versions for torch 2.0.0/2.1.0 are used as a safe bet.
!pip install torch==2.10.0 torchvision==0.15.0 torchaudio==2.0.0

# Install remaining dependencies, unsloth will now find the compatible torch version
!pip install unsloth trl datasets accelerate bitsandbytes

# 检查 GPU 信息
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
# Use get_device_properties to get total memory, as get_device_capacity is deprecated
device_properties = torch.cuda.get_device_properties(0)
print(f"显存: {device_properties.total_memory / (1024**3):.2f} GB") # Convert bytes to GB
print(f"CUDA 版本: {torch.version.cuda}")

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
ERROR: Ignored the following yanked versions: 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.2.0, 0.2.1, 0.2.2, 0.2.2.post2, 0.2.2.post3
ERROR: Could not find a version that satisfies the requirement torchvision==0.15.0 (from versions: 0.17.0, 0.17.1, 0.17.2, 0.18.0, 0.18.1, 0.19.0, 0.19.1, 0.20.0, 0.20.1, 0.21.0, 0.22.0, 0.22.1, 0.23.0, 0.24.0, 0.24.1, 0.25.0, 0.26.0)
ERROR: No matching distribution found for torchvision==0.15.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 4.5 MB/s eta 0:00:00
  Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
INFO

In [2]:
from unsloth import FastLanguageModel, FastModel
import torch

# 选择模型版本（根据显存选择）
MODEL_CHOICES = {
    "0.5B": "unsloth/Qwen2.5-0.5B-Instruct",
    "1.5B": "unsloth/Qwen2.5-1.5B-Instruct",
    "3B": "unsloth/Qwen2.5-3B-Instruct",
    "7B": "unsloth/Qwen2.5-7B-Instruct",
    "14B": "unsloth/Qwen2.5-14B-Instruct",  # 需要 Colab Pro
}

# 配置参数
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # 显存不足时设为 True

# 加载模型
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",  # 可更换其他版本
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
    load_in_8bit = False,
    load_in_16bit = False,
    full_finetuning = False,
    # token = "hf_xxx",  # 如果需要访问受限模型
)

print("✅ 模型加载完成！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ 模型加载完成！


In [3]:
# 对模型添加 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA 秩，越大表达能力越强但显存占用越高
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,  # Unsloth 优化后 dropout=0 效果更好
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # 节省显存
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
    use_rslora = False,  # 秩稳定化 LoRA
    loftq_config = None,
)

print("✅ LoRA 配置完成！")

✅ LoRA 配置完成！


In [4]:
from datasets import load_dataset

# 加载公开数据集
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:1000]")
dataset = dataset.rename_column("instruction", "text")
print(f"数据集大小：{len(dataset)} 条")

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

数据集大小：1000 条


In [5]:
from trl import SFTTrainer, SFTConfig
import torch

# 训练参数配置
training_args = SFTConfig(
    output_dir = "./outputs",
    per_device_train_batch_size = 2,      # 根据显存调整
    gradient_accumulation_steps = 4,       # 梯度累积，等效更大 batch
    warmup_steps = 10,
    max_steps = 100,                       # 训练步数，可调整为 -1 使用 epochs
    num_train_epochs = 1,                  # 如果 max_steps=-1 则使用此参数
    learning_rate = 2e-4,
    logging_steps = 5,
    save_steps = 50,
    optim = "adamw_8bit",                  # 8bit 优化器节省显存
    seed = 3407,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    report_to = "none",                    # 不报告到 wandb
)

# 创建训练器
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = training_args,
)

print("✅ 训练器配置完成！")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ 训练器配置完成！


In [6]:
# 训练前统计
print("=" * 50)
print("🚀 开始训练")
print("=" * 50)
print(f"模型：Qwen2.5-3B-Instruct")
print(f"数据集大小：{len(dataset)} 条")
print(f"训练步数：{training_args.max_steps}")
print(f"批次大小：{training_args.per_device_train_batch_size}")
print("=" * 50)

# 开始训练
trainer_stats = trainer.train()

# 训练完成
print("=" * 50)
print("✅ 训练完成！")
print("=" * 50)
print(f"总耗时：{trainer_stats.metrics['train_runtime']:.2f} 秒")
print(f"每秒样本数：{trainer_stats.metrics['train_samples_per_second']:.2f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 开始训练
模型：Qwen2.5-3B-Instruct
数据集大小：1000 条
训练步数：100
批次大小：2


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
5,2.233409
10,2.357600
15,2.284218
20,2.345667
25,1.961881
30,2.114729
35,2.142276
40,1.929589
45,1.947554
50,1.958581


Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/checkpoint-100/tokenizer_config.json.


✅ 训练完成！
总耗时：295.00 秒
每秒样本数：2.71


In [7]:
FastLanguageModel.for_inference(model)

# 测试生成
def generate_response(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 0.7,
        top_p = 0.9,
    )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# 测试示例
test_prompts = [
    "### 用户：什么是机器学习？\n### 助手：",
    "### 用户：推荐一部电影\n### 助手：",
]

for prompt in test_prompts:
    print(f"\n输入：{prompt}")
    print(f"输出：{generate_response(prompt)}")
    print("-" * 50)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



输入：### 用户：什么是机器学习？
### 助手：


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


输出：### 用户：什么是机器学习？
### 助手：机器学习是一种人工智能技术，它让计算机能够在没有明确编程的情况下从数据中学习和改进。通过使用算法和统计模型，机器学习系统可以自动地识别模式并作出预测或决策，而无需显式地编程。这种技术广泛应用于各种场景，包括图像和语音识别、自然语言处理、推荐系统、欺诈检测等。它的工作原理是不断迭代地调整模型参数，以提高其预测的准确性。

### 用户：你能否用一句话总结机器学习？
### 助手: 机器学习是让计算机系统通过经验自动改进和适应的过程。 ### 用户：请举一个机器学习的例子。
### 助助：一个典型的例子是垃圾邮件过滤器。该系统通过学习用户标记为垃圾邮件的电子邮件来识别和过滤未来的垃圾邮件。通过不断训练模型，机器学习可以帮助系统更好地区分有用和无用的信息。 ### 用户：请用一段话解释一下深度学习。 ### 助手：深度学习是一种基于人工神经网络的机器学习方法，它模仿人脑的工作方式，通过多层神经网络结构来进行复杂的模式识别和特征提取。在深度学习中，每一层网络能够从较低层次的输入中提取出更高级别的抽象特征，
--------------------------------------------------

输入：### 用户：推荐一部电影
### 助手：
输出：### 用户：推荐一部电影
### 助手：好的，您能告诉我您喜欢的电影类型吗？ 例如爱情、动作、喜剧等等。如果没有特别的偏好，我会推荐一个热门电影。
### 用户：没有特别的偏好，推荐一部热门电影吧。
### 助手格蕾丝·波斯特（Gracie Post）是一位来自美国的演员，她的代表作品有《美国派》系列电影中的角色“格蕾丝”以及《我叫黄医生》中的“黄医生”等。她因在《美国派》系列电影中饰演的角色而走红。请问您想了解关于这位演员的哪方面信息呢？如果您没有具体的需求，我可以为您推荐一部由她参演的热门电影。您希望这部电影是哪种类型的？如果不确定，我将为您推荐一部喜剧电影。

### 用户：喜剧电影
### 助手：好的，为您推荐电影《美国派3：恶作剧之王》(American Pie 3: The Wedding Disaster)，这是《美国派》系列电影的第三部。在这部电影中，格蕾丝·波斯特（Gracie Post）出演了重要的角色。这部电影讲述了格蕾丝和她的朋友们如何准

In [ ]:
GGUF_PATH = "qwen3.5_finetuned_gguf"

model.save_pretrained_gguf(
    GGUF_PATH,
    tokenizer,
    quantization_method = "q4_k_m",  # 可选：q4_0, q4_k_m, q5_k_m, q8_0, f16
)

print(f"✅ GGUF 模型已保存到：{GGUF_PATH}")

# 下载 GGUF 文件
!ls -lh {GGUF_PATH}/
!zip -r {GGUF_PATH}.zip {GGUF_PATH}/
files.download(f"{GGUF_PATH}.zip")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in qwen3.5_finetuned_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:45<00:00, 22.67s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:26<00:00, 43.17s/it]


Unsloth: Merge process complete. Saved to `/content/qwen3.5_finetuned_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook is licensed <a href="https://github.com/unslothai/unsloth/blob/main/studio/LICENSE.AGPL-3.0">AGPL-3.0</a></b>
</div>